# Olist Feature Advisor Example

This notebook uses the package notebook utilities to resolve the local `featurizer_config.yaml` workspace, then reads Olist metadata and runs the feature advisor. The advisor is executed in rule-based mode with a graph modeling intent.

In [2]:
import os
from pathlib import Path

import pandas as pd
from featurization.notebook_utils import build_notebook_resolver, get_featurization_artifact_paths
from featurization.feature_advisor_util import FeatureAdvisorUtil, FeatureAdvisorPromptConfig

# 1) Resolve the workspace using the notebook directory and KMDS working-dir conventions
cwd = Path.cwd()
if cwd.name == "notebooks" and (cwd.parent / "featurizer_config.yaml").exists():
    notebook_dir = cwd
elif (cwd / "notebooks").is_dir() and (cwd / "featurizer_config.yaml").exists():
    notebook_dir = cwd / "notebooks"
else:
    notebook_dir = cwd

resolver = build_notebook_resolver(str(notebook_dir))

print('Notebook dir:', notebook_dir)
print('Notebook dir exists:', notebook_dir.exists())
print('Config exists at parent:', (notebook_dir.parent / 'featurizer_config.yaml').exists())
print('Workspace root resolved from config parent:', resolver.working_dir)
print('Metadata path from config:', resolver.metadata_path)
print('Input data path from config:', resolver.featurization_input_path)

artifact_paths = get_featurization_artifact_paths(resolver)
print('Featurization artifact paths:')
for name, path in artifact_paths.items():
    print(f'- {name}: {path}')

# 2) Load Olist metadata anchored by the workspace config
metadata_path = Path(resolver.metadata_path)
if not metadata_path.exists():
    raise FileNotFoundError(
        f"Metadata file not found at {metadata_path}. "
        "Update `metadata_file` in featurizer_config.yaml or place the file there."
    )

metadata = pd.read_csv(metadata_path)
print('Metadata rows:', len(metadata))
print(metadata.head(3))

# 3) Build the advisor with the package prompt config
prompt_config = FeatureAdvisorPromptConfig.load_from_package()
advisor = FeatureAdvisorUtil(resolver=resolver, prompt_config=prompt_config)

print('Advisor output directory:', advisor.feature_advisor_dir)
print('Recommendation CSV path:', advisor.recommendations_csv_path)
print('Recommendation MD path:', advisor.recommendations_md_path)

# 4) Generate recommendations from the rule-based advisor
input_data_path = Path(resolver.featurization_input_path)
if not input_data_path.exists():
    raise FileNotFoundError(
        f"Input data file not found at {input_data_path}. "
        "Run the featurization pipeline or update `featurization_input_data` in featurizer_config.yaml."
    )

recommendations = advisor.recommend(
    metadata=metadata,
    model_intent='graph',
    input_data=pd.read_csv(input_data_path),
    use_rules=True,
)

print('Recommendations generated by rule-based advisor')
print(recommendations.head(10))
recommendations

Notebook dir: /home/rajiv/programming/kmds_migration/olist_migration/notebooks
Notebook dir exists: True
Config exists at parent: True
Workspace root resolved from config parent: /home/rajiv/programming/kmds_migration/olist_migration
Metadata path from config: /home/rajiv/programming/kmds_migration/olist_migration/data/../data_dictionary/data_dictionary.csv
Input data path from config: /home/rajiv/programming/kmds_migration/olist_migration/data/olist_daily_orders_prepared.csv
Featurization artifact paths:
- featurized_dataset_path: /home/rajiv/programming/kmds_migration/olist_migration/data/data/olist_daily_orders_prepared.csv
- model_ready_dataset_path: /home/rajiv/programming/kmds_migration/olist_migration/data/data/model_ready_numeric_data.csv
- feature_selection_knee_curve_path: /home/rajiv/programming/kmds_migration/olist_migration/data/data/feature_selection_knee_curve.png
Metadata rows: 14
                  attribute                                        description
0          

,attribute,recommended_method,rationale
0,order_id,low_count_cat_var_encoding + target_encoding,Use low-count categorical grouping on train da...
1,customer_id,low_count_cat_var_encoding + target_encoding,Use low-count categorical grouping on train da...
2,order_purchase_timestamp,low_count_cat_var_encoding + target_encoding,Use low-count categorical grouping on train da...
3,order_item_id,No encoding required,Numeric feature should be passed through the n...
4,product_id,low_count_cat_var_encoding + target_encoding,Use low-count categorical grouping on train da...
5,price,No encoding required,Numeric feature should be passed through the n...
6,customer_zip_code_prefix,No encoding required,Numeric feature should be passed through the n...
7,customer_city,low_count_cat_var_encoding + target_encoding,Use low-count categorical grouping on train da...
8,customer_state,low_count_cat_var_encoding + target_encoding,Use low-count categorical grouping on train da...
9,freq_cust,No encoding required,Numeric feature should be passed through the n...
